# 03 — DermLIP : contrôle de chevauchement, zéro-exemple et sonde linéaire

**Question** : un encodeur préentraîné sur des images de peau (DermLIP, Derm1M) voit-il mieux les lésions qu'un encodeur générique (ImageNet) ?

| ID | Expérience | Entraînement sur PAD-UFES-20 |
|---|---|---|
| — | Contrôle de chevauchement Derm1M / PAD-UFES-20 | — |
| E4 | DermLIP **zéro-exemple** (comparaison image / phrases) | aucun |
| E3 | DermLIP gelé + régression logistique (**sonde linéaire**) | logistique seulement |
| E2* | ViT-B/16 ImageNet gelé + sonde linéaire, **même protocole** que E3 | logistique seulement |

**Protocole** : mêmes 5 plis groupés par patient que les notebooks 01–02 ; le coefficient de régularisation `C` des sondes est choisi **à l'intérieur** de chaque pli d'entraînement par une validation croisée groupée par patient (validation croisée imbriquée), pour E2* comme pour E3. La comparaison E3 / E2* est donc équitable : même architecture (ViT-B/16), même protocole, seul le préentraînement change.

> **Avant d'exécuter**
> 1. Installer la bibliothèque : `pip install open_clip_torch`
> 2. La section 2 **télécharge les poids DermLIP** depuis Hugging Face (`redlessone/DermLIP_ViT-B-16`, environ 600 Mo, licence CC BY-NC 4.0 : recherche non commerciale).
> 3. Pour le contrôle de chevauchement, placer les CSV de Derm1M dans `data/external/derm1m/` (voir section 1). Sans eux, le contrôle est marqué « non vérifié ».

## 0. Imports, données et fonctions communes

In [ ]:
from pathlib import Path
import json
import re
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (balanced_accuracy_score, f1_score, recall_score,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
N_FOLDS = 5
np.random.seed(SEED)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CSV_PATH = ROOT / "zr7vgbcyr2-1" / "metadata.csv"
PROCESSED_DIR = ROOT / "data" / "processed"
EXTERNAL_DIR = ROOT / "data" / "external" / "derm1m"
CONFIG_DIR = ROOT / "configs"
FIG_DIR = ROOT / "outputs" / "figures"
REPORT_DIR = ROOT / "outputs" / "reports"
CKPT_DIR = ROOT / "outputs" / "checkpoints"
for d in (PROCESSED_DIR, FIG_DIR, REPORT_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# variables cliniques retenues (notebook 01) — le profil patient n'est utilisé que dans l'expérience E8
NUM_COLS = ["age"]
BOOL_COLS = ["itch", "grew", "hurt", "changed", "bleed", "elevation"]
CAT_COLS = ["region"]
FEATURES = NUM_COLS + BOOL_COLS + CAT_COLS
PROFILE_NUM = ["fitspatrick", "diameter_1", "diameter_2"]
PROFILE_BOOL = ["smoke", "drink", "pesticide", "skin_cancer_history", "cancer_history"]
PROFILE_CAT = ["gender"]
PROFILE_COLS = PROFILE_NUM + PROFILE_BOOL + PROFILE_CAT

CANCER_CLASSES = ["BCC", "SCC", "MEL"]
DERMLIP_ID = "hf-hub:redlessone/DermLIP_ViT-B-16"

In [ ]:
splits = pd.read_csv(PROCESSED_DIR / "splits.csv")
meta = pd.read_csv(CSV_PATH, na_values=["UNK"])

df = splits.merge(meta[["img_id"] + FEATURES + PROFILE_COLS], on="img_id", how="left",
                  validate="one_to_one")
TO_FLOAT = {True: 1.0, False: 0.0, "True": 1.0, "False": 0.0}
for c in BOOL_COLS + PROFILE_BOOL:                     # True / False / NaN -> 1.0 / 0.0 / NaN
    df[c] = df[c].map(TO_FLOAT)

dev = df[df["split"] == "train"].reset_index(drop=True)
test = df[df["split"] == "test"].reset_index(drop=True)
y_dev, folds, groups_dev = dev["label"].to_numpy(), dev["fold"].to_numpy(), dev["patient_id"].to_numpy()
y_test, groups_test = test["label"].to_numpy(), test["patient_id"].to_numpy()

print(f"Développement : {len(dev)} images, {dev['patient_id'].nunique()} patients | "
      f"test : {len(test)} images (non utilisé avant l'évaluation finale)")
assert set(folds) == set(range(N_FOLDS))

In [ ]:
def metrics_at(y_true, proba, threshold=0.5):
    y_pred = (proba >= threshold).astype(int)
    return {
        "AUC": roc_auc_score(y_true, proba),
        "Exactitude équilibrée": balanced_accuracy_score(y_true, y_pred),
        "Sensibilité": recall_score(y_true, y_pred, pos_label=1),
        "Spécificité": recall_score(y_true, y_pred, pos_label=0),
        "F1": f1_score(y_true, y_pred),
    }


def take_rows(X, idx):
    return X.iloc[idx] if isinstance(X, pd.DataFrame) else X[idx]


def fit_model(model, X, y, groups=None):
    """Entraîne un modèle ; transmet `groups` aux modèles qui choisissent leurs hyperparamètres par patient."""
    needs_groups = isinstance(model, GridSearchCV) or getattr(model, "uses_groups", False)
    if needs_groups:
        assert groups is not None, "ce modèle a besoin des identifiants patients (groups)"
        return model.fit(X, y, groups=groups)
    return model.fit(X, y)


def cross_validate(make_model, X, y, folds, groups=None, return_models=False):
    """Validation croisée sur les 5 plis groupés par patient : métriques par pli + prédictions hors pli."""
    oof, rows, models = np.zeros(len(y)), [], []
    for k in range(N_FOLDS):
        tr, va = np.flatnonzero(folds != k), np.flatnonzero(folds == k)
        model = make_model()
        fit_model(model, take_rows(X, tr), y[tr], None if groups is None else groups[tr])
        oof[va] = model.predict_proba(take_rows(X, va))[:, 1]
        rows.append({"pli": k, **metrics_at(y[va], oof[va])})
        models.append(model)
    per_fold = pd.DataFrame(rows)
    return (per_fold, oof, models) if return_models else (per_fold, oof)


def summarize(name, per_fold):
    m = per_fold.drop(columns="pli")
    return {"Expérience": name, **{c: f"{m[c].mean():.3f} ± {m[c].std():.3f}" for c in m.columns},
            "AUC moyenne": m["AUC"].mean()}


def grouped_bootstrap(y, p, groups, p_ref=None, n_boot=2000, seed=SEED):
    """AUC et IC 95 % par bootstrap des patients ; avec `p_ref`, différence d'AUC appariée."""
    y, p, groups = np.asarray(y), np.asarray(p), np.asarray(groups)
    uniq, inv = np.unique(groups, return_inverse=True)
    members = [np.flatnonzero(inv == g) for g in range(len(uniq))]
    rng = np.random.default_rng(seed)
    aucs, diffs = [], []
    for _ in range(n_boot):
        idx = np.concatenate([members[g] for g in rng.integers(0, len(uniq), len(uniq))])
        if y[idx].min() == y[idx].max():
            continue
        a = roc_auc_score(y[idx], p[idx])
        aucs.append(a)
        if p_ref is not None:
            diffs.append(a - roc_auc_score(y[idx], np.asarray(p_ref)[idx]))
    out = {"AUC": roc_auc_score(y, p), "IC95 bas": np.percentile(aucs, 2.5), "IC95 haut": np.percentile(aucs, 97.5)}
    if p_ref is not None:
        d = np.asarray(diffs)
        out.update({"ΔAUC": out["AUC"] - roc_auc_score(y, p_ref),
                    "ΔAUC IC95 bas": np.percentile(d, 2.5), "ΔAUC IC95 haut": np.percentile(d, 97.5),
                    "P(Δ ≤ 0)": float((d <= 0).mean())})
    return out


def make_clinical_preprocessor(num=NUM_COLS, bools=BOOL_COLS, cats=CAT_COLS):
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num),
        ("bool", SimpleImputer(strategy="most_frequent"), bools),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cats),
    ])


C_GRID = [0.003, 0.01, 0.03, 0.1, 0.3, 1.0]


def make_probe(c_grid=C_GRID):
    """Sonde linéaire : standardisation + logistique, C choisi par validation croisée groupée (patients)."""
    pipe = Pipeline([("scale", StandardScaler()),
                     ("clf", LogisticRegression(class_weight="balanced", max_iter=5000))])
    return GridSearchCV(pipe, {"clf__C": c_grid}, cv=GroupKFold(n_splits=3), scoring="roc_auc", n_jobs=-1)


def load_features(name):
    """Charge un cache de caractéristiques et renvoie une fonction frame -> matrice alignée sur frame['img_id']."""
    feats = np.load(PROCESSED_DIR / f"features_{name}.npy")
    ids = pd.read_csv(PROCESSED_DIR / f"features_{name}_ids.csv")["img_id"]
    index = pd.Series(np.arange(len(ids)), index=ids.to_numpy())
    return lambda frame: feats[index.loc[frame["img_id"]].to_numpy()]


results, oof_preds = [], {}

## 1. Contrôle de chevauchement Derm1M / PAD-UFES-20

Derm1M contient des images issues de « datasets publics ». Si des images de PAD-UFES-20 ont servi au préentraînement de DermLIP, nos scores seraient optimistes. Deux contrôles :

1. **Texte et noms de fichiers** : recherche, dans tous les CSV de Derm1M, de mentions de PAD-UFES-20 et de noms au format PAD (`PAT_<patient>_<lésion>_<n>`).
2. **Hachage perceptuel** (optionnel, si les images de Derm1M sont téléchargées) : un « dHash » de 64 bits résume chaque image ; deux images presque identiques (même redimensionnées ou recompressées) ont des hachages proches (distance de Hamming ≤ 6).

Fichiers attendus dans `data/external/derm1m/` : `Derm1M_v2_pretrain.csv`, `Derm1M_v2_validation.csv` (et, pour le contrôle 2, les images dans `data/external/derm1m/images/`).

In [ ]:
PAD_NAME = re.compile(r"PAT_\d+_\d+_\d+", re.IGNORECASE)
PAD_TEXT = re.compile(r"pad[-_ ]?ufes|ufes[-_ ]?20", re.IGNORECASE)
our_ids = {i.lower(): i for i in df["img_id"]}

overlap = {"statut": "non vérifié", "fichiers": [], "mentions_PAD_texte": 0, "noms_PAD_trouves": 0}
overlap_rows = []
csv_files = sorted(p for p in EXTERNAL_DIR.glob("*.csv") if "concept" not in p.name.lower())

if not csv_files:
    print("Aucun CSV Derm1M dans", EXTERNAL_DIR)
    print("→ Contrôle NON VÉRIFIÉ : à mentionner comme limite dans le rapport.")
else:
    found = set()
    for f in csv_files:
        n_rows = n_text = 0
        for chunk in pd.read_csv(f, chunksize=100_000, dtype=str, on_bad_lines="skip"):
            n_rows += len(chunk)
            text = chunk.fillna("").agg(" ".join, axis=1)
            n_text += int(text.str.contains(PAD_TEXT).sum())
            for name in text.str.findall(PAD_NAME).explode().dropna():
                found.add(name.lower() + ".png")
        overlap["fichiers"].append({"fichier": f.name, "lignes": n_rows, "mentions_PAD": n_text})
        overlap["mentions_PAD_texte"] += n_text
        print(f"{f.name} : {n_rows} lignes, {n_text} mentions de PAD-UFES-20")
    common = sorted(our_ids[n] for n in found if n in our_ids)
    overlap["noms_PAD_trouves"] = len(common)
    overlap_rows += [{"img_id": i, "methode": "nom de fichier", "distance": 0} for i in common]
    overlap["statut"] = "vérifié (texte et noms)"
    print("Images PAD-UFES-20 retrouvées par leur nom :", len(common))
    print("Colonnes du premier fichier :", list(pd.read_csv(csv_files[0], nrows=0).columns))

In [ ]:
RUN_HASH_CHECK = False          # passer à True si les images de Derm1M sont dans data/external/derm1m/images/
DERM1M_IMG_DIR = EXTERNAL_DIR / "images"
MAX_DERM1M_IMAGES = 200_000     # limite de temps de calcul
HAMMING_MAX = 6
POPCOUNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


def dhash(path, size=8):
    with Image.open(path) as im:
        g = np.asarray(im.convert("L").resize((size + 1, size), Image.Resampling.LANCZOS), dtype=np.int16)
    return np.packbits((g[:, 1:] > g[:, :-1]).ravel()).view(">u8")[0].astype(np.uint64)


def nearest_hamming(a, b, chunk=2048):
    """Pour chaque hachage de `a`, distance de Hamming minimale vers `b` et indice correspondant."""
    best, arg = np.full(len(a), 65), np.full(len(a), -1)
    for s in range(0, len(b), chunk):
        x = np.bitwise_xor(a[:, None], b[None, s:s + chunk])
        d = POPCOUNT[x.view(np.uint8).reshape(len(a), -1, 8)].sum(axis=-1)
        j = d.argmin(axis=1)
        dj = d[np.arange(len(a)), j]
        better = dj < best
        best[better], arg[better] = dj[better], j[better] + s
    return best, arg


if RUN_HASH_CHECK and DERM1M_IMG_DIR.exists():
    derm_paths = [p for p in DERM1M_IMG_DIR.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    derm_paths = derm_paths[:MAX_DERM1M_IMAGES]
    t0 = time.time()
    pad_hash = np.array([dhash(ROOT / p) for p in df["img_path"]], dtype=np.uint64)
    derm_hash = np.array([dhash(p) for p in derm_paths], dtype=np.uint64)
    dist, arg = nearest_hamming(pad_hash, derm_hash)
    hits = np.flatnonzero(dist <= HAMMING_MAX)
    overlap_rows += [{"img_id": df["img_id"].iloc[i], "methode": "hachage perceptuel",
                      "distance": int(dist[i]), "image_derm1m": str(derm_paths[arg[i]])} for i in hits]
    overlap["hachage"] = {"images_derm1m_comparees": len(derm_paths), "doublons_probables": int(len(hits)),
                          "seuil_hamming": HAMMING_MAX, "duree_s": round(time.time() - t0)}
    print(overlap["hachage"])
else:
    overlap["hachage"] = "non exécuté"
    print("Contrôle par hachage non exécuté (RUN_HASH_CHECK = False ou images absentes).")

overlap_df = pd.DataFrame(overlap_rows, columns=["img_id", "methode", "distance"]).drop_duplicates("img_id")
overlap_df.to_csv(PROCESSED_DIR / "derm1m_overlap_ids.csv", index=False)
overlap["images_concernees"] = int(len(overlap_df))
overlap["dont_test"] = int(overlap_df["img_id"].isin(test["img_id"]).sum())
(REPORT_DIR / "03_controle_chevauchement.json").write_text(json.dumps(overlap, indent=2, ensure_ascii=False), encoding="utf8")
print(json.dumps(overlap, indent=2, ensure_ascii=False))

**Règle de décision** (fixée à l'avance) : les images retrouvées sont listées dans `data/processed/derm1m_overlap_ids.csv`. Le notebook 05 rapportera les résultats du test **avec et sans** ces images ; si elles sont nombreuses, les résultats DermLIP doivent être présentés comme potentiellement optimistes.

## 2. Chargement de DermLIP

> Cette cellule télécharge les poids au premier lancement (≈ 600 Mo, mis en cache par Hugging Face ensuite).

In [ ]:
try:
    import open_clip
    import torch
except ImportError as e:
    raise ImportError("Installez open_clip :  pip install open_clip_torch") from e

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
model, _, preprocess = open_clip.create_model_and_transforms(DERMLIP_ID)
tokenizer = open_clip.get_tokenizer(DERMLIP_ID)
model = model.to(device).eval()
LOGIT_SCALE = float(model.logit_scale.exp().item())
print("DermLIP chargé sur", device, "| échelle des logits :", round(LOGIT_SCALE, 1))


@torch.no_grad()
def encode_texts(texts):
    emb = model.encode_text(tokenizer(texts).to(device)).float()
    return torch.nn.functional.normalize(emb, dim=-1).cpu().numpy()


def prompt_embedding(prompts):
    """Moyenne des plongements de plusieurs formulations, renormalisée (« ensemble de prompts »)."""
    e = encode_texts(prompts).mean(axis=0)
    return e / np.linalg.norm(e)


@torch.no_grad()
def encode_tensors(x):
    emb = model.encode_image(x.to(device)).float()
    return torch.nn.functional.normalize(emb, dim=-1).cpu().numpy()


def load_tensor(path):
    with Image.open(path) as im:
        return preprocess(im.convert("RGB"))

## 3. Extraction des plongements d'images (cache)

Chaque image est convertie en RGB, redimensionnée et recadrée à 224 × 224 par le prétraitement officiel de DermLIP, puis encodée en un vecteur de 512 valeurs normalisé (norme 1). Le résultat est mis en cache : l'extraction n'est faite qu'une fois.

In [ ]:
FEAT_PATH = PROCESSED_DIR / "features_dermlip_vitb16.npy"
IDS_PATH = PROCESSED_DIR / "features_dermlip_vitb16_ids.csv"
BATCH = 64 if device == "cuda" else 32

if FEAT_PATH.exists() and IDS_PATH.exists():
    print("Cache trouvé :", np.load(FEAT_PATH).shape)
else:
    paths, feats, t0 = (ROOT / df["img_path"]).tolist(), [], time.time()
    for start in range(0, len(paths), BATCH):
        feats.append(encode_tensors(torch.stack([load_tensor(p) for p in paths[start:start + BATCH]])))
        if (start // BATCH) % 10 == 0:
            print(f"{min(start + BATCH, len(paths))}/{len(paths)} images — {time.time() - t0:.0f} s")
    np.save(FEAT_PATH, np.concatenate(feats).astype(np.float32))
    df[["img_id"]].to_csv(IDS_PATH, index=False)
    print("Plongements enregistrés en", round(time.time() - t0), "s")

take_dermlip = load_features("dermlip_vitb16")
X_dermlip_dev = take_dermlip(dev)
print("Matrice de développement :", X_dermlip_dev.shape,
      "| normes ≈ 1 :", np.allclose(np.linalg.norm(X_dermlip_dev, axis=1), 1, atol=1e-3))

## 4. E4 — Zéro-exemple

Aucun entraînement : on compare chaque image à des phrases décrivant les 6 diagnostics. Chaque diagnostic est décrit par plusieurs formulations dont on moyenne les plongements (« ensemble de prompts »). La probabilité de cancer est la somme des probabilités de BCC, SCC et MEL.

In [ ]:
CLASS_TEXT = {
    "BCC": "basal cell carcinoma",
    "SCC": "squamous cell carcinoma",
    "MEL": "melanoma",
    "ACK": "actinic keratosis",
    "NEV": "melanocytic nevus",
    "SEK": "seborrheic keratosis",
}
TEMPLATES = [
    "a clinical photo of {}.",
    "a close-up clinical photo of {} on the skin.",
    "a smartphone photo of a skin lesion: {}.",
    "{}.",
]
classes = list(CLASS_TEXT)
class_emb = np.stack([prompt_embedding([t.format(CLASS_TEXT[c]) for t in TEMPLATES]) for c in classes])


def zero_shot_proba(emb):
    logits = LOGIT_SCALE * emb @ class_emb.T
    logits -= logits.max(axis=1, keepdims=True)
    p = np.exp(logits)
    return p / p.sum(axis=1, keepdims=True)


P6 = zero_shot_proba(X_dermlip_dev)
p_zs = P6[:, [classes.index(c) for c in CANCER_CLASSES]].sum(axis=1)
oof_preds["E4 DermLIP zéro-exemple"] = p_zs

rows = [{"pli": k, **metrics_at(y_dev[folds == k], p_zs[folds == k])} for k in range(N_FOLDS)]
results.append(summarize("E4 DermLIP zéro-exemple", pd.DataFrame(rows)))

pred6 = np.array(classes)[P6.argmax(axis=1)]
print("Exactitude équilibrée (6 classes) :", round(balanced_accuracy_score(dev["diagnostic"], pred6), 3))
pd.crosstab(dev["diagnostic"], pred6, rownames=["vrai"], colnames=["prédit (zéro-exemple)"])

## 5. E3 et E2* — Sondes linéaires (validation croisée imbriquée)

Pour chaque pli : `C` est choisi parmi 6 valeurs par une validation croisée à 3 plis **groupée par patient** sur les seules données d'entraînement du pli, puis le modèle est évalué sur le pli de validation. E2* reprend les caractéristiques ImageNet du notebook 02 si elles existent.

In [ ]:
experiments = {"E3 DermLIP sonde linéaire": X_dermlip_dev}
if (PROCESSED_DIR / "features_imagenet_vitb16.npy").exists():
    experiments["E2* ImageNet sonde linéaire"] = load_features("imagenet_vitb16")(dev)
else:
    print("Caractéristiques ImageNet absentes : exécutez la section 6 du notebook 02 pour obtenir E2*.")

best_c = {}
for name, X in experiments.items():
    t0 = time.time()
    per_fold, oof_preds[name], models = cross_validate(make_probe, X, y_dev, folds, groups=groups_dev,
                                                       return_models=True)
    results.append(summarize(name, per_fold))
    best_c[name] = [m.best_params_["clf__C"] for m in models]
    print(f"{name} — {time.time() - t0:.0f} s — C choisi par pli : {best_c[name]}")
    print(per_fold.round(3).to_string(index=False), "\n")

## 6. Référence clinique (E1a, identique au notebook 02)

In [ ]:
def make_clinical_logreg():
    return Pipeline([("prep", make_clinical_preprocessor()),
                     ("clf", LogisticRegression(class_weight="balanced", max_iter=2000))])


per_fold, oof_preds["E1a Clinique – logistique"] = cross_validate(make_clinical_logreg, dev[FEATURES], y_dev, folds)
results.append(summarize("E1a Clinique – logistique", per_fold))

## 7. Comparaison et tests statistiques

In [ ]:
table = pd.DataFrame(results).set_index("Expérience").sort_values("AUC moyenne", ascending=False)
table.drop(columns="AUC moyenne").to_csv(REPORT_DIR / "03_dermlip_resultats.csv")
table.drop(columns="AUC moyenne")

In [ ]:
comparisons = [("E3 DermLIP sonde linéaire", "E2* ImageNet sonde linéaire"),
               ("E3 DermLIP sonde linéaire", "E4 DermLIP zéro-exemple"),
               ("E3 DermLIP sonde linéaire", "E1a Clinique – logistique")]
rows = []
for a, b in comparisons:
    if a in oof_preds and b in oof_preds:
        r = grouped_bootstrap(y_dev, oof_preds[a], groups_dev, p_ref=oof_preds[b])
        rows.append({"comparaison": f"{a}  vs  {b}", **{k: round(v, 3) for k, v in r.items()}})
stats = pd.DataFrame(rows)
stats.to_csv(REPORT_DIR / "03_comparaisons_bootstrap.csv", index=False)
stats

*Lecture* : `ΔAUC` est la différence d'AUC (hors pli) entre les deux modèles ; l'intervalle `IC95` vient de 2 000 tirages bootstrap **des patients** (un patient et toutes ses images sont tirés ensemble). Si l'intervalle ne contient pas 0, la différence est considérée comme réelle.

In [ ]:
plt.figure(figsize=(6.5, 6))
for name, proba in oof_preds.items():
    fpr, tpr, _ = roc_curve(y_dev, proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y_dev, proba):.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=0.8)
plt.xlabel("1 − spécificité")
plt.ylabel("Sensibilité")
plt.title("Courbes ROC hors pli — encodeurs d'images")
plt.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig(FIG_DIR / "03_courbes_roc_dermlip.png", dpi=150)
plt.show()

## 8. Sauvegarde

In [ ]:
oof = dev[["img_id", "patient_id", "diagnostic", "label", "fold"]].copy()
for name, proba in oof_preds.items():
    oof[name] = proba
oof.to_csv(PROCESSED_DIR / "oof_dermlip.csv", index=False)
(REPORT_DIR / "03_C_choisis.json").write_text(json.dumps(best_c, indent=2, ensure_ascii=False), encoding="utf8")
print("Enregistré :", PROCESSED_DIR / "oof_dermlip.csv")

## Conclusion (à compléter après exécution)

Grille de lecture fixée à l'avance :

- **E3 vs E2*** : si l'IC 95 % de ΔAUC est entièrement > 0, le préentraînement dermatologique (Derm1M) apporte un gain réel → DermLIP est retenu comme encodeur d'images. Sinon, on le conserve pour les concepts (notebook 04) mais le gain de vision n'est pas démontré.
- **E4** : un zéro-exemple nettement au-dessus de 0,5 confirme que DermLIP « comprend » les diagnostics sans entraînement ; l'écart E3 − E4 mesure l'apport de l'apprentissage sur PAD-UFES-20.
- **E3 vs E1a** : indique si l'image seule rivalise avec les symptômes ; la fusion (notebook 05) doit faire mieux que les deux.
- **Chevauchement** : reporter le statut et le nombre d'images concernées (fichier `03_controle_chevauchement.json`).

**Étape suivante** : `04_concepts.ipynb`.